# Línea Base para Predicción de Combates Pokémon

### Importación de Módulos

In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from scipy.sparse import csr_matrix

import warnings
warnings.filterwarnings("ignore")

RANDOM_STATE = 42

### Carga de Datos

In [8]:
df_cleaned = pd.read_csv("../data/gen9ou_cleaned_dataset.csv")
df_cleaned.head()

,battle_id,p1_poke1,p1_poke2,p1_poke3,p1_poke4,p1_poke5,p1_poke6,p2_poke1,p2_poke2,p2_poke3,p2_poke4,p2_poke5,p2_poke6,p1_win
0,1636793-gen9ou-2460221181,Dragapult,Kingambit,Hawlucha,Glimmora,Chesnaught,Gallade,Porygon-Z,Great Tusk,Raging Bolt,Corviknight,Meowscarada,Dragonite,1
1,1636793-gen9ou-2460221181,Porygon-Z,Great Tusk,Raging Bolt,Corviknight,Meowscarada,Dragonite,Dragapult,Kingambit,Hawlucha,Glimmora,Chesnaught,Gallade,0
2,95801-gen9ou-2513923983,Meganium,Arboliva,Avalugg,Ditto,Houndoom,Tinkaton,Hawlucha,Rillaboom,Azumarill,Iron Moth,Corviknight,Raging Bolt,1
3,95801-gen9ou-2513923983,Hawlucha,Rillaboom,Azumarill,Iron Moth,Corviknight,Raging Bolt,Meganium,Arboliva,Avalugg,Ditto,Houndoom,Tinkaton,0
4,1514712-gen9ou-2429668049,Dragonite,Zamazenta,LandorusTherianTherian,Gholdengo,Darkrai,Hatterene,Ninetales,Ceruledge,Venusaur,Hatterene,Great Tusk,Walking Wake,1


## 1. Separación de Conjuntos de Entrenamiento y Prueba

Split según battle_id para evitar data leakage

In [9]:
battle_ids = df_cleaned['battle_id'].unique()

train_ids, test_ids = train_test_split(
    battle_ids,
    test_size=0.2,
    random_state=RANDOM_STATE
)

train_df = df_cleaned[df_cleaned['battle_id'].isin(train_ids)]
test_df  = df_cleaned[df_cleaned['battle_id'].isin(test_ids)]

In [10]:
X_train = train_df.drop(['p1_win', 'battle_id'], axis=1) 
Y_train = train_df['p1_win']

X_test = test_df.drop(['p1_win', 'battle_id'], axis=1)
Y_test = test_df['p1_win']

## 2. Encoding

In [11]:
pokemon_cols = [
    "p1_poke1","p1_poke2","p1_poke3","p1_poke4","p1_poke5","p1_poke6",
    "p2_poke1","p2_poke2","p2_poke3","p2_poke4","p2_poke5","p2_poke6"
]

all_pokemon = pd.unique(df_cleaned[pokemon_cols].values.ravel())

pokemon_to_idx = {p: i for i, p in enumerate(sorted(all_pokemon))}
n_pokemon = len(pokemon_to_idx)

print("Número de Pokémon:", n_pokemon)

Número de Pokémon: 809


Se codifica el dataset utilizando Multi-hot encoding (manual) con una representación de 1 para cada Pokémon presente en el combate y 0 para los ausentes. Esto se hace para ambos equipos (equipo 1 y equipo 2) y se concatenan las representaciones para formar la matriz de características final.

Además, se utiliza `scipy.sparse.csr_matrix` para convertir la matriz de características a un formato disperso, lo que es eficiente en términos de memoria dado que la mayoría de los valores serán ceros.

In [ ]:
def build_sparse_matrix(df, pokemon_to_idx):

    rows = []
    cols = []
    
    y = df["p1_win"].values
    
    n_pokemon = len(pokemon_to_idx)
    
    for row_idx, row in enumerate(df.itertuples(index=False)):
        
        # equipo jugador 1
        for p in [row.p1_poke1,row.p1_poke2,row.p1_poke3,row.p1_poke4,row.p1_poke5,row.p1_poke6]:
            cols.append(pokemon_to_idx[p])
            rows.append(row_idx)
        
        # equipo jugador 2
        for p in [row.p2_poke1,row.p2_poke2,row.p2_poke3,row.p2_poke4,row.p2_poke5,row.p2_poke6]:
            cols.append(n_pokemon + pokemon_to_idx[p])
            rows.append(row_idx)

    data = np.ones(len(rows), dtype=np.int8)

    X = csr_matrix(
        (data, (rows, cols)),
        shape=(len(df), 2 * n_pokemon),
        dtype=np.int8
    )

    return X, y

In [13]:
X_train, y_train = build_sparse_matrix(train_df, pokemon_to_idx)
X_test, y_test = build_sparse_matrix(test_df, pokemon_to_idx)

print(X_train.shape)

(2818934, 1618)


A continuación, se selecciona una muestra de 100000 combates para entrenar el modelo de baseline, debido a limitaciones de memoria y tiempo de entrenamiento.

In [14]:
RF_SAMPLE_BATTLES = 100_000

rf_battle_ids = np.random.choice(
    train_df["battle_id"].unique(),
    size=RF_SAMPLE_BATTLES,
    replace=False
)

rf_df = train_df[train_df["battle_id"].isin(rf_battle_ids)]

In [15]:
X_rf, y_rf = build_sparse_matrix(rf_df, pokemon_to_idx)

## 3. Entrenamiento del Modelo de Baseline

### 3.1 Entrenamiento de Random Forest

Ya que Random Forest no puede manejar matrices dispersas, convertimos a formato denso (aunque esto puede consumir mucha memoria, por lo que se recomienda hacerlo solo con una muestra de los datos).

In [17]:
X_rf_dense = X_rf.toarray()
X_test_dense = X_test.toarray()

In [18]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    n_jobs=-1,
    random_state=RANDOM_STATE
)

rf_model.fit(X_rf_dense, y_rf)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",300
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric

In [19]:
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score

rf_pred = rf_model.predict(X_test_dense)
rf_proba = rf_model.predict_proba(X_test_dense)[:,1]

rf_acc = accuracy_score(y_test, rf_pred)
rf_auc = roc_auc_score(y_test, rf_proba)
rf_f1 = f1_score(y_test, rf_pred)

print("Random Forest results")
print("Accuracy:", rf_acc)
print("ROC-AUC:", rf_auc)
print("F1:", rf_f1)

Random Forest results
Accuracy: 0.5410296650934963
ROC-AUC: 0.5590402645705485
F1: 0.5371534217093665


### 3.2 Entrenamiento de LightGBM

Dado que LightGBM puede manejar matrices dispersas, se entrena directamente con la matriz en formato CSR sin necesidad de convertir a formato denso, lo que es más eficiente en términos de memoria. Por lo tanto, se puede entrenar con un mayor número de muestras sin preocuparse por el consumo de memoria.

In [21]:
import lightgbm as lgb

lgb_model = lgb.LGBMClassifier(
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=64,
    n_jobs=-1,
    random_state=RANDOM_STATE
)

lgb_model.fit(X_train, y_train)

[LightGBM] [Info] Number of positive: 1409467, number of negative: 1409467
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 2.503514 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2968
[LightGBM] [Info] Number of data points in the train set: 2818934, number of used features: 1484
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000


,boosting_type,'gbdt'
,num_leaves,64
,max_depth,-1
,learning_rate,0.05
,n_estimators,500
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [22]:
lgb_pred = lgb_model.predict(X_test)
lgb_proba = lgb_model.predict_proba(X_test)[:,1]

lgb_acc = accuracy_score(y_test, lgb_pred)
lgb_auc = roc_auc_score(y_test, lgb_proba)
lgb_f1 = f1_score(y_test, lgb_pred)

print("LightGBM results")
print("Accuracy:", lgb_acc)
print("ROC-AUC:", lgb_auc)
print("F1:", lgb_f1)

LightGBM results
Accuracy: 0.5551683330164289
ROC-AUC: 0.5808226236880538
F1: 0.5553967119370979


## 4. Representación Team Differential

Se basa en la idea de representar cada combate como la diferencia entre los pokémon del equipo 1 y los pokémon del equipo 2. Esto se hace restando la representación de los pokémon del equipo 2 de la representación de los pokémon del equipo 1 de la forma `team_diff_vector = team1_vector - team2_vector`, lo que da como resultado una nueva representación que captura las diferencias entre ambos equipos. 

In [23]:
def build_team_differential(df, pokemon_to_idx):
    n_pokemon = len(pokemon_to_idx)
    
    rows = []
    cols = []
    data = []
    
    y = df['p1_win'].values
    
    for row_idx, row in enumerate(df.itertuples(index=False)):
        # team 1: +1
        for p in [row.p1_poke1,row.p1_poke2,row.p1_poke3,row.p1_poke4,row.p1_poke5,row.p1_poke6]:
            cols.append(pokemon_to_idx[p])
            rows.append(row_idx)
            data.append(1)
        
        # team 2: -1
        for p in [row.p2_poke1,row.p2_poke2,row.p2_poke3,row.p2_poke4,row.p2_poke5,row.p2_poke6]:
            cols.append(pokemon_to_idx[p])
            rows.append(row_idx)
            data.append(-1)
    
    X = csr_matrix((data, (rows, cols)), shape=(len(df), n_pokemon), dtype=np.int8)
    
    return X, y

In [24]:
# Train completo
X_train_diff, y_train = build_team_differential(train_df, pokemon_to_idx)
X_test_diff, y_test = build_team_differential(test_df, pokemon_to_idx)

print("Team Differential shapes:")
print("X_train:", X_train_diff.shape)
print("X_test :", X_test_diff.shape)

Team Differential shapes:
X_train: (2818934, 809)
X_test : (704734, 809)


Al igual que antes, se selecciona una muestra de 100000 combates para entrenar el modelo de baseline con esta nueva representación.

In [25]:
RF_SAMPLE_BATTLES = 100_000  # ajustar según memoria

rf_battle_ids = np.random.choice(
    train_df["battle_id"].unique(),
    size=RF_SAMPLE_BATTLES,
    replace=False
)

rf_df = train_df[train_df["battle_id"].isin(rf_battle_ids)]

X_rf_diff, y_rf = build_team_differential(rf_df, pokemon_to_idx)

# Random Forest necesita matriz densa
X_rf_diff_dense = X_rf_diff.toarray()
X_test_diff_dense = X_test_diff.toarray()

## 5. Entrenamiento de Modelos con Representación Team Differential

### 5.1 Entrenamiento de Random Forest con Team Differential

In [26]:
rf_model_diff = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    n_jobs=-1,
    random_state=RANDOM_STATE
)

rf_model_diff.fit(X_rf_diff_dense, y_rf)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",300
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric

### 5.2 Entrenamiento de LightGBM con Team Differential

In [28]:
X_train_diff = X_train_diff.astype('float32')
X_test_diff = X_test_diff.astype('float32')

In [29]:
import lightgbm as lgb

lgb_model_diff = lgb.LGBMClassifier(
    objective="binary",
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=64,
    feature_fraction=0.8,
    bagging_fraction=0.8,
    bagging_freq=5,
    n_jobs=-1,
    random_state=RANDOM_STATE
)

lgb_model_diff.fit(X_train_diff, y_train)

[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] bagging_freq is set=5, subsample_freq=0 will be ignored. Current value: bagging_freq=5
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] bagging_freq is set=5, subsample_freq=0 will be ignored. Current value: bagging_freq=5
[LightGBM] [Info] Number of positive: 1409467, number of negative: 1409467
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 1.325205 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2233
[LightGBM] [Info] Number of dat

,boosting_type,'gbdt'
,num_leaves,64
,max_depth,-1
,learning_rate,0.05
,n_estimators,500
,subsample_for_bin,200000
,objective,'binary'
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


### 5.3 Evaluación de Modelos con Team Differential

In [30]:
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score

# Random Forest
rf_pred = rf_model_diff.predict(X_test_diff_dense)
rf_proba = rf_model_diff.predict_proba(X_test_diff_dense)[:,1]

rf_acc = accuracy_score(y_test, rf_pred)
rf_auc = roc_auc_score(y_test, rf_proba)
rf_f1 = f1_score(y_test, rf_pred)

print("Random Forest (Team Differential)")
print("Accuracy:", rf_acc)
print("ROC-AUC:", rf_auc)
print("F1:", rf_f1)

# LightGBM
lgb_pred = lgb_model_diff.predict(X_test_diff)
lgb_proba = lgb_model_diff.predict_proba(X_test_diff)[:,1]

lgb_acc = accuracy_score(y_test, lgb_pred)
lgb_auc = roc_auc_score(y_test, lgb_proba)
lgb_f1 = f1_score(y_test, lgb_pred)

print("LightGBM (Team Differential)")
print("Accuracy:", lgb_acc)
print("ROC-AUC:", lgb_auc)
print("F1:", lgb_f1)

Random Forest (Team Differential)
Accuracy: 0.5395212945593657
ROC-AUC: 0.5562531377004758
F1: 0.5354973812996597
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] bagging_freq is set=5, subsample_freq=0 will be ignored. Current value: bagging_freq=5
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] bagging_freq is set=5, subsample_freq=0 will be ignored. Current value: bagging_freq=5
LightGBM (Team Differential)
Accuracy: 0.553379005411971
ROC-AUC: 0.578304896268504
F1: 0.5534898830912927


In [31]:
import pandas as pd

results_diff = pd.DataFrame({
    "model": ["RandomForest", "LightGBM"],
    "representation": ["team_differential", "team_differential"],
    "accuracy": [rf_acc, lgb_acc],
    "roc_auc": [rf_auc, lgb_auc],
    "f1": [rf_f1, lgb_f1]
})

results_diff

,model,representation,accuracy,roc_auc,f1
0,RandomForest,team_differential,0.539521,0.556253,0.535497
1,LightGBM,team_differential,0.553379,0.578305,0.553490
